# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a demonstration of how to load and explore the FAIR² dataset using the `mlcroissant` library. The dataset covers clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors, including MSI-H status and anatomical distribution.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records from the FAIR² dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}:")
print(f"{metadata.description}\n")

## 2. Data Overview

Review available record sets, including their `@id`, fields, columns, and types as defined by the Croissant schema.

In [ ]:
# List all record sets in the dataset, identified by their `@id`
record_sets = dataset.record_sets
print("Available Record Sets (@id):\n--------------------------")
for rs in record_sets:
    print(f"- {rs['@id']} ({rs.get('name','--no name--')})")

# Explore fields inside each record set
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    print("  Fields:")
    for field in rs.get('field', []):
        print(f"    - @id: {field['@id']}")
        print(f"      name: {field.get('name','--no name--')}")
        print(f"      dataType: {field.get('dataType','--')}\n")

## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s gathered in the previous overview.

**Note:** The main dataset is typically the largest record set with the bulk of patient records. Based on the Croissant schema for this dataset, the principal record set (for the data table of cancer survivors with clinicopathological variables) usually has an `@id` like `cr:recordSet` or a longer @id such as `https://sen.science/doi/10.71728/senscience.qs2f-h81p/recordSet/data`.
For demonstration purposes, we'll use the first available record set.

In [ ]:
# Extract data for each record set found above
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dfs = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dfs[record_set_id] = df
            print(f"Loaded {len(df)} records. Columns: {list(df.columns)}\n")
        else:
            print("No records found for this set.\n")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}\n")

# Choose the main record set by number of columns
main_record_set_id = max(dfs, key=lambda k: len(dfs[k].columns))
main_df = dfs[main_record_set_id]
print(f"\nMain Record Set ID: {main_record_set_id}")
print("Fields (columns):")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering, normalizing a numeric field, and grouping data by a categorical variable. All field and column references are by their `@id` for clarity and consistency.

In [ ]:
# List all available columns for reference by @id
print("Available columns in main record set DataFrame:")
print(list(main_df.columns))

# Example: select a numeric field, e.g., 'age_at_second_crc_diagnosis' (replace with actual @id if different)
# For demonstration we will use the first numeric-looking column found
numeric_field_id = None
for col in main_df.columns:
    if main_df[col].dtype in ['int64', 'float64']:
        numeric_field_id = col
        break

if numeric_field_id is not None:
    print(f"\nSelected numeric field for analysis: {numeric_field_id}")

    threshold = main_df[numeric_field_id].quantile(0.25)
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No numeric field found for demonstration.")

# Now group by a categorical field, for example, anatomical location or MSI status (again, by @id)
# Let's auto-select the first non-numeric column for grouping
group_field_id = None
for col in main_df.columns:
    if main_df[col].dtype == object and col != numeric_field_id:
        group_field_id = col
        break

if numeric_field_id is not None and group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id}, showing mean of {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to the chosen grouping field. For demonstration, a histogram and boxplot are used.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in main_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to load, inspect, and perform basic analysis on a FAIR²-compliant dataset using `mlcroissant`.
- All navigation of the schema and data extraction was handled using `@id` references for record sets and fields.
- Typical EDA steps included filtering records, normalizing a numeric field, grouping and summarizing data, and visualization.

For further investigation, the analyst can use the loaded DataFrames and schema metadata to conduct custom queries, export to CSV, and prepare ML pipelines.